In [6]:
import pandas as pd
import random
import string
import uuid
from deltalake import DeltaTable, write_deltalake

In [10]:
view1_filename = './data/tlhop-epss-dashboard-view1.delta'
view2a_filename = './data/tlhop-epss-dashboard-view2a.delta'
view2b_filename = './data/tlhop-epss-dashboard-view2b.delta'
view3_filename = './data/tlhop-epss-dashboard-view3.delta'

In [2]:
def gen_text(size=28):
    return str(uuid.uuid4())[0:size]

def gen_integer(minv=1, maxv=9999):
    return random.randint(minv, maxv)

def gen_float(max_v=10.0):
    return random.uniform(0.1, max_v)

def gen_by_option(chars):
    return random.choice(chars)

def gen_ip():
    return ".".join(map(str, (random.randint(0, 255) for _ in range(4))))

def bucket_epss(score):

    if score < 0.2:
        return "< 0.2"
    elif score < 0.4:
        return "< 0.4"
    elif score < 0.6:
        return "< 0.6"
    elif score < 0.8:
        return "< 0.8"
    else:
        return ">= 0.8"

def bucket_cvss(score):
    if score:
        if score < 0.1:
            return "None"
        elif score < 4.0:
            return "low"
        elif score < 7.0:
            return "medium"
        elif score < 9.0:
            return "high"
        else: 
            return "critical"
    else:
        return "None"

In [3]:
df_v1 = pd.DataFrame([
    ["< 0.2",3061,  5448,  39216],
    ["< 0.4",51,    2630,  10243],
    ["< 0.6",35,    3382,  17175],
    ["< 0.8",32,    2936,  13483],
    [">= 0.8",142,  3206,  15316]
], columns=["epss_rank","n_cves","n_orgs","n_ips"])
df_v1

,epss_rank,n_cves,n_orgs,n_ips
0,< 0.2,3061,5448,39216
1,< 0.4,51,2630,10243
2,< 0.6,35,3382,17175
3,< 0.8,32,2936,13483
4,>= 0.8,142,3206,15316


In [7]:
write_deltalake(view1_filename, df_v1, mode='append')

In [8]:
n_size_cves = 100

cves = {}
for _ in range(n_size_cves):
    cve_id = "CVE-%s-%s" % (gen_integer(2019, 2024), gen_text(4))
    epss = round(gen_float(1.0), 2)
    cvss = round(gen_float(10.0), 1)
    b_epss = bucket_epss(epss)
    b_cvss = bucket_cvss(cvss)
    cves[cve_id] = {'epss': epss, 'b_epss': b_epss, 'cvss': cvss, 'b_cvss': b_cvss}

In [12]:
df_v2a = []
n_size_row = 1000


for _ in range(n_size_row):
    cve_id = gen_by_option(list(cves.keys()))
    row = [gen_text(), # org_clean
           gen_ip(), # ip_str
           gen_text(8),  # cpe_product
           str(round(gen_float(5.0), 1)), # cpe_version
           cve_id, # cve_id
           cves[cve_id]['epss'], # epss
           cves[cve_id]['b_epss'], # epss_rank
           cves[cve_id]['cvss'],  # cvss
           cves[cve_id]['b_cvss'],  # cvss_rank
           gen_by_option(['2.0', '3.1']) # cvss_version
          ]
    df_v2a.append(row)
    
df_v2a = pd.DataFrame(df_v2a, columns=["org_clean", 'ip_str', 'cpe_product', 'cpe_version', 'cve_id', 'epss', 'epss_rank', 'cvss_score','cvss_rank', 'cvss_version'])
df_v2a

,org_clean,ip_str,cpe_product,cpe_version,cve_id,epss,epss_rank,cvss_score,cvss_rank,cvss_version
0,8224021c-c4c6-4c08-8fd8-dd85,138.101.108.42,e5689e60,3.5,CVE-2021-4a9c,0.95,>= 0.8,5.2,medium,2.0
1,a525f8d6-f49b-4f64-868b-4335,189.185.109.18,8fd33e1b,2.5,CVE-2024-17ea,0.52,< 0.6,8.2,high,2.0
2,1b7d23c7-6f1e-4009-b694-27e8,198.63.178.91,fe755aa1,3.2,CVE-2019-335e,0.17,< 0.2,3.5,low,2.0
3,0b99794c-8ba0-401e-92f2-b846,194.243.62.190,49bf1a23,4.8,CVE-2024-8b03,0.98,>= 0.8,3.5,low,3.1
4,1c2c658e-6d45-496b-a192-f246,149.94.88.241,d4179239,4.6,CVE-2022-fe34,0.12,< 0.2,6.2,medium,2.0
...,...,...,...,...,...,...,...,...,...,...
995,a90cea21-6481-46d0-9d95-5712,29.230.125.199,5654ea08,2.4,CVE-2021-473c,0.62,< 0.8,9.6,critical,2.0
996,1e2e12a4-0011-4f8e-b143-4905,81.144.144.183,2155f033,4.3,CVE-2019-d0fd,0.92,>= 0.8,0.7,low,3.1
997,28fa3dab-a138-4f55-8a78-80a2,4.51.78.129,055a34aa,2.5,CVE-2019-4768,0.24,< 0.4,3.9,low,2.0
998,3fbc1c53-d10c-4648-8b33-6443,211.40.27.78,7fa0329a,1.9,CVE-2023-8696,0.59,< 0.6,8.2,high,2.0


In [13]:
write_deltalake(view2a_filename, df_v2a, mode='append')

In [16]:
df_v2b = []
n_size_row = 1000

for _ in range(n_size_row):
    
    n_cves = gen_integer(1, 10)
    n_ips = gen_integer(1, 10)
    n_cpes = gen_integer(1, 10)
    
    cve_list = [gen_by_option(list(cves.keys())) for _ in range(n_cves)]
    ips_list = [gen_ip() for _ in range(n_ips)]
    cpe_list = [gen_text(8) for _ in range(n_cpes)]
    
    row = [gen_text(), # org_clean
           cves[cve_id]['epss'], # epss
           cves[cve_id]['b_epss'], # epss_rank
           "; ".join(cve_list),
           "; ".join(ips_list),
           "; ".join(cve_list),
          ]
    df_v2b.append(row)
    
df_v2b = pd.DataFrame(df_v2b, columns=["org_clean", 'epss_major', 'epss_rank_major', 'cpe_list', 'ip_list', 'cve_list'])
df_v2b

,org_clean,epss_major,epss_rank_major,cpe_list,ip_list,cve_list
0,8f9f61da-1b04-4c45-8800-63db,0.66,< 0.8,CVE-2019-5b8e; CVE-2019-a71b; CVE-2023-e4b1; C...,143.159.58.112; 45.81.16.100; 23.129.134.235; ...,CVE-2019-5b8e; CVE-2019-a71b; CVE-2023-e4b1; C...
1,368d932b-1211-4cdd-8ff7-7de9,0.66,< 0.8,CVE-2020-39f4; CVE-2023-b2b6; CVE-2024-12b7; C...,168.157.81.151; 205.254.17.120; 251.199.246.23...,CVE-2020-39f4; CVE-2023-b2b6; CVE-2024-12b7; C...
2,85948154-c214-4cac-852a-dbad,0.66,< 0.8,CVE-2024-8b03; CVE-2023-63aa; CVE-2021-0f52,243.70.49.163; 166.99.70.83; 66.242.61.240; 21...,CVE-2024-8b03; CVE-2023-63aa; CVE-2021-0f52
3,eb891940-d48e-44df-8011-6176,0.66,< 0.8,CVE-2022-e13a; CVE-2022-c528; CVE-2019-6615; C...,90.252.88.48; 142.217.157.165; 49.69.102.143; ...,CVE-2022-e13a; CVE-2022-c528; CVE-2019-6615; C...
4,d5e6b600-6f87-4946-bda0-4eb6,0.66,< 0.8,CVE-2023-3ad4,177.225.174.227; 188.251.202.50; 141.141.104.1...,CVE-2023-3ad4
...,...,...,...,...,...,...
995,4ce52bc1-9355-42c7-a675-3649,0.66,< 0.8,CVE-2020-d8af; CVE-2020-f023; CVE-2023-63aa; C...,150.99.112.150,CVE-2020-d8af; CVE-2020-f023; CVE-2023-63aa; C...
996,5c00017c-5100-4112-a80e-438a,0.66,< 0.8,CVE-2020-7aa8; CVE-2020-560c; CVE-2019-b4e2; C...,80.249.57.23; 236.215.253.106; 134.81.211.241;...,CVE-2020-7aa8; CVE-2020-560c; CVE-2019-b4e2; C...
997,07f2b5ec-d59a-4117-8693-5546,0.66,< 0.8,CVE-2020-d8af; CVE-2019-cc52; CVE-2019-1ad7; C...,6.112.34.101; 104.120.213.122; 170.156.23.71; ...,CVE-2020-d8af; CVE-2019-cc52; CVE-2019-1ad7; C...
998,5d5f762d-47f5-47d2-87b4-2390,0.66,< 0.8,CVE-2022-c528; CVE-2024-a408; CVE-2019-cc52; C...,252.45.173.192; 95.41.72.97; 107.215.126.171; ...,CVE-2022-c528; CVE-2024-a408; CVE-2019-cc52; C...


In [17]:
write_deltalake(view2b_filename, df_v2b, mode='append')

In [20]:
df_v3 = []
n_size_row = 1000


for _ in range(n_size_row):
    
    cve_id = gen_by_option(list(cves.keys()))
    n_orgs = gen_integer(1, 1000)
    orgs_list = [gen_text(8) for _ in range(n_orgs)]
    
    row = [cve_id,
           cves[cve_id]['epss'], # epss
           cves[cve_id]['b_epss'], # epss_rank
           cves[cve_id]['cvss'],
           cves[cve_id]['b_cvss'],
           gen_by_option(['2.0', '3.1']),
           n_orgs,
           gen_integer(2, 100),
           # orgs_list
          ]
    df_v3.append(row)
    
df_v3 = pd.DataFrame(df_v3, columns=['cve_id', 'epss', "epss_rank",   'cvss_score', 'cvss_rank', 'cvss_version', 'n_orgs','n_ips'])
df_v3

,cve_id,epss,epss_rank,cvss_score,cvss_rank,cvss_version,n_orgs,n_ips
0,CVE-2019-335e,0.17,< 0.2,3.5,low,2.0,44,48
1,CVE-2019-d0fd,0.92,>= 0.8,0.7,low,2.0,407,13
2,CVE-2023-741c,0.46,< 0.6,6.0,medium,2.0,6,43
3,CVE-2024-a408,0.56,< 0.6,1.9,low,2.0,39,68
4,CVE-2019-f381,0.20,< 0.4,8.8,high,2.0,273,37
...,...,...,...,...,...,...,...,...
995,CVE-2020-d8af,0.74,< 0.8,9.3,critical,3.1,845,19
996,CVE-2024-cfac,0.97,>= 0.8,9.3,critical,2.0,923,31
997,CVE-2022-c528,0.14,< 0.2,6.5,medium,2.0,704,20
998,CVE-2019-313f,0.80,>= 0.8,2.7,low,3.1,351,39


In [21]:
write_deltalake(view3_filename, df_v3, mode='append')

In [22]:
DeltaTable(view3_filename).history()

[{'timestamp': 1714824866664,
  'operation': 'WRITE',
  'operationParameters': {'partitionBy': '[]', 'mode': 'Append'},
  'clientVersion': 'delta-rs.0.17.3',
  'version': 1},
 {'timestamp': 1714780369604,
  'operation': 'WRITE',
  'operationParameters': {'partitionBy': '[]', 'mode': 'Append'},
  'isolationLevel': 'Serializable',
  'isBlindAppend': True,
  'engineInfo': 'Apache-Spark/3.4.1 Delta-Lake/3.0.0rc1',
  'txnId': '298cd8f4-4c4b-4c9d-80ed-11323e61333c',
  'operationMetrics': {'numFiles': '1',
   'numOutputBytes': '45307',
   'numOutputRows': '4270'},
  'version': 0}]